# Week 4 Workshop — Notebook 2: Coordinate Transformation and Multiplicative Gain

**Theoretical Neuroscience — University of Copenhagen**

In this notebook you will explore the **Pouget & Sejnowski basis-function model** for coordinate transformation in the posterior parietal cortex (PPC).

Each PPC neuron responds as the **product** of:
- A Gaussian **retinotopic tuning curve** $F_k(x)$ — how the neuron responds to retinal image position $x$
- A sigmoidal **gain field** $G_k(y)$ — how the response is modulated by gaze direction $y$

$$r_k(x, y) = F_k(x) \cdot G_k(y)$$

with

$$F_k(x) = A \, \exp\!\left(-\frac{(x - x_k)^2}{2\sigma^2}\right)$$

$$G_k(y) = \frac{1}{1 + \exp\!\left(-\dfrac{y - y_k}{\lambda}\right)}$$

A linear readout of these basis functions can compute the **head-centred location** $z = x + y$.

---

## 0. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from ipywidgets import interact, FloatSlider, IntSlider

print('All imports successful.')

---
## 1. Define the tuning curve components

We define $F_k(x)$ (Gaussian, retinal position) and $G_k(y)$ (sigmoid, gaze direction) as standalone functions.

In [ ]:
def F(x, x_k, sigma, A=1.0):
    """Gaussian retinotopic tuning curve.

    Parameters
    ----------
    x     : array-like, retinal position (degrees)
    x_k   : float, preferred retinal position of neuron k
    sigma : float, tuning width (degrees)
    A     : float, amplitude (max firing rate)
    """
    return A * np.exp(-((x - x_k)**2) / (2 * sigma**2))


def G(y, y_k, lam):
    """Sigmoidal gain field.

    Parameters
    ----------
    y     : array-like, gaze direction (degrees)
    y_k   : float, inflection point of neuron k's gain field
    lam   : float, slope parameter (degrees); larger = shallower slope
    """
    return 1.0 / (1.0 + np.exp(-(y - y_k) / lam))


def basis_function(x, y, x_k, y_k, sigma, lam, A=1.0):
    """Full 2D basis function r_k(x, y) = F_k(x) * G_k(y)."""
    return F(x, x_k, sigma, A) * G(y, y_k, lam)

---
## 2. Plot $F_k(x)$ and $G_k(y)$ individually

Before combining them, let's visualise each component separately.

In [ ]:
# ── Axes ───────────────────────────────────────────────────────────────────────
x = np.linspace(-30, 30, 400)   # retinal position (degrees)
y = np.linspace(-30, 30, 400)   # gaze direction (degrees)

# ── Parameters for one example neuron ─────────────────────────────────────────
x_k   = 5.0    # preferred retinal position
y_k   = 0.0    # gain field inflection point
sigma = 8.0    # Gaussian width
lam   = 5.0    # sigmoid slope
A     = 1.0    # amplitude

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# F_k(x)
axes[0].plot(x, F(x, x_k, sigma, A), color='steelblue', linewidth=2)
axes[0].axvline(x_k, color='steelblue', linestyle='--', alpha=0.6, label=f'$x_k = {x_k}°$')
axes[0].set_xlabel('Retinal position $x$ (°)', fontsize=12)
axes[0].set_ylabel('$F_k(x)$', fontsize=12)
axes[0].set_title(f'Gaussian tuning curve ($\\sigma = {sigma}°$)', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].set_ylim([-0.05, 1.15])

# G_k(y)
axes[1].plot(y, G(y, y_k, lam), color='darkorange', linewidth=2)
axes[1].axvline(y_k, color='darkorange', linestyle='--', alpha=0.6, label=f'$y_k = {y_k}°$')
axes[1].set_xlabel('Gaze direction $y$ (°)', fontsize=12)
axes[1].set_ylabel('$G_k(y)$', fontsize=12)
axes[1].set_title(f'Sigmoid gain field ($\\lambda = {lam}°$)', fontsize=13)
axes[1].legend(fontsize=11)
axes[1].set_ylim([-0.05, 1.15])

plt.suptitle('Individual tuning curve components for neuron $k$', fontsize=14)
plt.tight_layout()
plt.show()

**Questions:**
- What does $\sigma$ control about $F_k(x)$? What happens to the tuning curve as $\sigma \to 0$ or $\sigma \to \infty$?
- What does $\lambda$ control about $G_k(y)$? What is the shape of $G_k$ when $\lambda$ is very small vs. very large?

---
## 3. The 2D basis function $r_k(x, y) = F_k(x) \cdot G_k(y)$

The product of the Gaussian and the sigmoid produces a 2D response field — a tuning curve over the joint space of retinal position and gaze direction.

In [ ]:
XX, YY = np.meshgrid(x, y)   # shape: (n_y, n_x)
R = basis_function(XX, YY, x_k=5.0, y_k=0.0, sigma=8.0, lam=5.0, A=1.0)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Filled contour
cf = axes[0].contourf(XX, YY, R, levels=40, cmap='hot')
plt.colorbar(cf, ax=axes[0], label='$r_k(x,y)$')
axes[0].set_xlabel('Retinal position $x$ (°)', fontsize=12)
axes[0].set_ylabel('Gaze direction $y$ (°)', fontsize=12)
axes[0].set_title('Basis function $r_k(x,y) = F_k(x) \\cdot G_k(y)$', fontsize=13)

# 3D surface
ax3d = fig.add_subplot(1, 2, 2, projection='3d', computed_zorder=False)
ax3d.plot_surface(XX, YY, R, cmap='hot', linewidth=0, antialiased=True, alpha=0.9)
ax3d.set_xlabel('$x$ (°)', fontsize=10)
ax3d.set_ylabel('$y$ (°)', fontsize=10)
ax3d.set_zlabel('$r_k$', fontsize=10)
ax3d.set_title('3D view of basis function', fontsize=13)

plt.tight_layout()
plt.show()

---
## 4. A population of PPC neurons

We now create a population of $K$ neurons with **different** preferred positions $x_k$ and gain inflection points $y_k$, tiling the $(x, y)$ space.

In [ ]:
# ── Population parameters ──────────────────────────────────────────────────────
n_xk   = 7     # number of different preferred retinal positions
n_yk   = 5     # number of different gain field centres
K      = n_xk * n_yk   # total number of neurons

sigma  = 8.0   # Gaussian width (degrees)
lam    = 5.0   # sigmoid slope (degrees)
A      = 1.0   # amplitude

x_ks = np.linspace(-20, 20, n_xk)   # preferred retinal positions
y_ks = np.linspace(-20, 20, n_yk)   # gain field centres

# Build the full response matrix for all (x, y) pairs in the grid
# R_pop[k, iy, ix] = response of neuron k to stimulus at (x[ix], y[iy])
R_pop = np.zeros((K, len(y), len(x)))
for idx, (xk, yk) in enumerate([(xk, yk) for yk in y_ks for xk in x_ks]):
    R_pop[idx] = basis_function(XX, YY, xk, yk, sigma, lam, A)

print(f'Population of K = {K} neurons created.')

In [ ]:
# ── Plot a grid of individual basis functions ──────────────────────────────────
fig, axes = plt.subplots(n_yk, n_xk, figsize=(14, 8),
                          sharex=True, sharey=True)

for i_y, yk in enumerate(y_ks):
    for i_x, xk in enumerate(x_ks):
        ax = axes[i_y, i_x]
        R_ij = basis_function(XX, YY, xk, yk, sigma, lam, A)
        ax.contourf(x, y, R_ij, levels=15, cmap='hot')
        ax.set_xticks([]); ax.set_yticks([])
        if i_x == 0:
            ax.set_ylabel(f'$y_k={yk:.0f}°$', fontsize=8)
        if i_y == n_yk - 1:
            ax.set_xlabel(f'$x_k={xk:.0f}°$', fontsize=8)

plt.suptitle(f'Population of {K} basis functions tiling the (x, y) space\n'
             f'$\\sigma={sigma}°$, $\\lambda={lam}°$', fontsize=13)
plt.tight_layout()
plt.show()

**Questions:**
- How do the basis functions change as you move across rows (varying $y_k$) or columns (varying $x_k$)?
- Do you see any redundancy or gaps in the tiling? How would you adjust $\sigma$ and $\lambda$ to avoid gaps?

---
## 5. Head-centred tuning curves at fixed gaze

If we fix the gaze direction $y$ and vary retinal position $x$, the response of each neuron traces out a **modulated Gaussian**: $r_k(x, y^*) = G_k(y^*) \cdot F_k(x)$. The gain factor $G_k(y^*)$ simply scales the peak but does not shift it.

In [ ]:
gaze_angles = [-15, 0, 15]   # fixed gaze directions to compare
x_k_example = 0.0             # one neuron with x_k at centre
y_k_example = 0.0

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

for ax, y_fixed in zip(axes, gaze_angles):
    gain_val = G(y_fixed, y_k_example, lam)
    r_curve  = F(x, x_k_example, sigma, A) * gain_val

    ax.plot(x, r_curve, color='steelblue', linewidth=2,
            label=f'$r_k(x, {y_fixed}°)$')
    ax.plot(x, F(x, x_k_example, sigma, A), color='grey', linewidth=1.2,
            linestyle='--', label='$F_k(x)$ (no gain)')
    ax.axhline(gain_val, color='darkorange', linestyle=':', linewidth=1.5,
               label=f'$G_k({y_fixed}°) = {gain_val:.2f}$')
    ax.set_xlabel('Retinal position $x$ (°)', fontsize=12)
    ax.set_title(f'Gaze $y = {y_fixed}°$', fontsize=12)
    ax.legend(fontsize=9)
    ax.set_ylim([-0.05, 1.1])

axes[0].set_ylabel('Firing rate (normalised)', fontsize=12)
plt.suptitle('Tuning curves at fixed gaze angles — gain modulation scales the peak',
             fontsize=13)
plt.tight_layout()
plt.show()

---
## 6. Linear readout: decoding head-centred position

A downstream neuron can decode the head-centred position $z = x + y$ by taking a **weighted sum** of the PPC basis functions:

$$\hat{z} = \sum_k w_k \, r_k(x, y)$$

We fit these weights with linear regression and check how well $\hat{z}$ matches the true $z = x + y$.

In [ ]:
from sklearn.linear_model import Ridge

# ── Build a training dataset ───────────────────────────────────────────────────
n_train = 2000
rng_fit = np.random.default_rng(0)
x_train = rng_fit.uniform(-20, 20, n_train)
y_train = rng_fit.uniform(-20, 20, n_train)
z_train = x_train + y_train    # target: head-centred position

# Population response matrix: shape (n_train, K)
def pop_response(x_vals, y_vals, x_ks, y_ks, sigma, lam, A=1.0):
    R = np.zeros((len(x_vals), len(x_ks) * len(y_ks)))
    for idx, (xk, yk) in enumerate([(xk, yk) for yk in y_ks for xk in x_ks]):
        R[:, idx] = basis_function(x_vals, y_vals, xk, yk, sigma, lam, A)
    return R

Phi_train = pop_response(x_train, y_train, x_ks, y_ks, sigma, lam)

# ── Fit linear readout ─────────────────────────────────────────────────────────
reg = Ridge(alpha=1e-3)
reg.fit(Phi_train, z_train)

# ── Evaluate on test set ───────────────────────────────────────────────────────
n_test  = 500
x_test  = rng_fit.uniform(-20, 20, n_test)
y_test  = rng_fit.uniform(-20, 20, n_test)
z_test  = x_test + y_test
Phi_test = pop_response(x_test, y_test, x_ks, y_ks, sigma, lam)
z_hat   = reg.predict(Phi_test)

r2 = reg.score(Phi_test, z_test)
print(f'Linear readout R² on test set: {r2:.4f}')

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(z_test, z_hat, alpha=0.4, s=10, color='steelblue')
ax.plot([-40, 40], [-40, 40], 'r--', linewidth=1.5, label='Identity')
ax.set_xlabel('True $z = x + y$ (°)', fontsize=12)
ax.set_ylabel('Decoded $\\hat{z}$ (°)', fontsize=12)
ax.set_title(f'Linear readout of head-centred position\n$R^2 = {r2:.3f}$', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

**Questions:**
- How accurate is the linear readout ($R^2$ close to 1)? What does this tell you about the computational power of the basis function representation?
- Would the readout still work if you used only $F_k(x)$ without the gain field $G_k(y)$? Why or why not?

---
## 7. Interactive exploration: vary $\sigma$, $\lambda$, $x_k$, $y_k$

Use the sliders to explore how each parameter shapes the basis function.

In [ ]:
def plot_basis(x_k=5.0, y_k=0.0, sigma=8.0, lam=5.0, A=1.0):
    fig = plt.figure(figsize=(15, 4))
    gs  = GridSpec(1, 4, figure=fig)

    # F_k(x)
    ax1 = fig.add_subplot(gs[0])
    ax1.plot(x, F(x, x_k, sigma, A), color='steelblue', lw=2)
    ax1.axvline(x_k, color='steelblue', ls='--', alpha=0.6)
    ax1.set_xlabel('$x$ (°)'); ax1.set_ylabel('$F_k(x)$')
    ax1.set_title(f'Gaussian\n$x_k={x_k}°$, $\\sigma={sigma}°$')
    ax1.set_ylim(-0.05, A*1.15)

    # G_k(y)
    ax2 = fig.add_subplot(gs[1])
    ax2.plot(y, G(y, y_k, lam), color='darkorange', lw=2)
    ax2.axvline(y_k, color='darkorange', ls='--', alpha=0.6)
    ax2.set_xlabel('$y$ (°)'); ax2.set_ylabel('$G_k(y)$')
    ax2.set_title(f'Sigmoid\n$y_k={y_k}°$, $\\lambda={lam}°$')
    ax2.set_ylim(-0.05, 1.15)

    # 2D basis function — contour
    R_ij = basis_function(XX, YY, x_k, y_k, sigma, lam, A)
    ax3 = fig.add_subplot(gs[2])
    cf = ax3.contourf(x, y, R_ij, levels=30, cmap='hot')
    plt.colorbar(cf, ax=ax3)
    ax3.set_xlabel('$x$ (°)'); ax3.set_ylabel('$y$ (°)')
    ax3.set_title('Basis function\n$r_k(x,y) = F_k \\cdot G_k$')

    # 3D surface
    ax4 = fig.add_subplot(gs[3], projection='3d')
    ax4.plot_surface(XX, YY, R_ij, cmap='hot', linewidth=0, alpha=0.9)
    ax4.set_xlabel('$x$', fontsize=9); ax4.set_ylabel('$y$', fontsize=9)
    ax4.set_zlabel('$r_k$', fontsize=9)
    ax4.set_title('3D view')

    plt.suptitle('Basis function components — interactive', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()


interact(plot_basis,
         x_k  = FloatSlider(value=5.0,  min=-25.0, max=25.0, step=1.0,  description='$x_k$ (°)'),
         y_k  = FloatSlider(value=0.0,  min=-25.0, max=25.0, step=1.0,  description='$y_k$ (°)'),
         sigma= FloatSlider(value=8.0,  min=1.0,   max=25.0, step=0.5,  description='$\\sigma$ (°)'),
         lam  = FloatSlider(value=5.0,  min=0.5,   max=20.0, step=0.5,  description='$\\lambda$ (°)'),
         A    = FloatSlider(value=1.0,  min=0.1,   max=3.0,  step=0.1,  description='$A$'));

**Guided exploration tasks:**

1. Set $\lambda$ very small (e.g. 0.5). What does the sigmoid look like? What kind of neuron does this resemble?
2. Set $\lambda$ very large (e.g. 20). How does the gain field change, and how does this affect the 2D basis function?
3. Set $\sigma$ very small. How does the Gaussian tuning curve change?
4. Move $x_k$ and $y_k$ to different positions. How does the location of the 2D response field shift?
5. What combination of $\sigma$ and $\lambda$ gives the most separable (factored) appearance in the 2D contour plot?

---
## 8. Effect of $\sigma$ and $\lambda$ on the population and readout

Finally, we test how systematically changing $\sigma$ and $\lambda$ affects how well the population encodes $z = x + y$.

In [ ]:
sigmas = [3, 6, 10, 16]
lambdas = [2, 5, 10, 18]

R2_grid = np.zeros((len(sigmas), len(lambdas)))

for i, sig in enumerate(sigmas):
    for j, lm in enumerate(lambdas):
        Phi_tr = pop_response(x_train, y_train, x_ks, y_ks, sig, lm)
        Phi_te = pop_response(x_test,  y_test,  x_ks, y_ks, sig, lm)
        reg_ij = Ridge(alpha=1e-3).fit(Phi_tr, z_train)
        R2_grid[i, j] = reg_ij.score(Phi_te, z_test)

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(R2_grid, origin='lower', cmap='RdYlGn', vmin=0, vmax=1,
               aspect='auto')
plt.colorbar(im, ax=ax, label='$R^2$ (linear readout)')
ax.set_xticks(range(len(lambdas))); ax.set_xticklabels(lambdas)
ax.set_yticks(range(len(sigmas)));  ax.set_yticklabels(sigmas)
ax.set_xlabel('$\\lambda$ (sigmoid slope, °)', fontsize=12)
ax.set_ylabel('$\\sigma$ (Gaussian width, °)', fontsize=12)
ax.set_title('Readout quality as a function of tuning parameters', fontsize=13)

for i in range(len(sigmas)):
    for j in range(len(lambdas)):
        ax.text(j, i, f'{R2_grid[i, j]:.2f}', ha='center', va='center',
                fontsize=11, color='black')

plt.tight_layout()
plt.show()

**Questions:**
- In which regime of $\sigma$ and $\lambda$ does the readout perform best? Why?
- What happens when $\sigma$ is very narrow? What about very broad?
- Very small $\lambda$ makes $G_k$ nearly a step function. Does this help or hurt the readout? Why?
- Based on these results, what would be the "optimal" tuning parameters for PPC neurons that need to support coordinate transformation?

---
*End of Notebook 2*